# שלב 3 — השוואה בין השיטות (Benchmark)

טוען את הפלטים והמטריקות של כל השיטות שמומשו ומפיק טבלאות השוואה למצגת המסכמת:

1. **סקירת שיטות** — מבנה נתונים, מבוזר?, זמן ריצה, זיכרון, מס' תוצאות
2. **תוצאות לפי קונפיגורציה** — כמה תתי-מסלולים פופולריים לכל ספֵּף ק"מ
3. **סטטיסטיקות אורך/תמיכה** — max/mean/median/p90 ק"מ, תמיכה
4. **הסכמה בין השיטות** — Jaccard, precision, recall, F1 מול שיטת ייחוס מדויקת
5. **דיוק מול משאבים** — trade-off

מריצים אחרי `03a` / `03b` (/ `03c`). שיטות שחסר להן פלט פשוט מדולגות.


## 1. טעינה

In [ ]:
import os, json, numpy as np, pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("compare").master("local[*]").config("spark.driver.memory","4g").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

CONFIGS_KM = [1, 3, 5, 10, 20, 40]
REFERENCE = "graph_cms"     # שיטת ייחוס (מדויקת) למדדי הסכמה

# (method -> (results_parquet, metrics_json)). הוסיפו כאן את שיטת ה-clustering כשתהיה.
CANDIDATES = {
    "graph_cms":     ("out/subroutes_graph_sample.parquet",     "out/metrics_graph.json"),
    "suffix_hll":    ("out/subroutes_suffix_sample.parquet",    "out/metrics_suffix.json"),
    "clustering_lsh":("out/subroutes_clustering_sample.parquet","out/metrics_clustering.json"),
}

dfs, metrics = {}, {}
for m, (pq, mj) in CANDIDATES.items():
    if os.path.exists(mj) and os.path.exists(pq):
        dfs[m] = spark.read.parquet(pq).select("win_key","n_cells","support","km").toPandas()
        metrics[m] = json.load(open(mj))
print("loaded methods:", list(dfs.keys()))


## 2. סקירת שיטות + Speedup

In [ ]:
ov = pd.DataFrame([{
    "method": m, "structure": metrics[m]["structure"], "distributed": metrics[m]["distributed"],
    "runtime_sec": metrics[m]["runtime_sec"], "struct_mem_MB": round(metrics[m]["struct_mem_bytes"]/1e6, 2),
    "n_maximal": metrics[m]["n_maximal"],
} for m in dfs])
fastest = min(metrics, key=lambda m: metrics[m]["runtime_sec"])
ov["speedup_vs_slowest"] = (ov["runtime_sec"].max() / ov["runtime_sec"]).round(2)
display(ov)
print("fastest method:", fastest)


## 3. תוצאות לפי קונפיגורציה (#תתי-מסלולים פופולריים ≥ K ק"מ)

In [ ]:
pc = pd.DataFrame({m: {k: metrics[m]["per_config"].get(str(k), 0) for k in CONFIGS_KM} for m in dfs})
pc.index.name = "min_km"
display(pc)


## 4. סטטיסטיקות אורך ותמיכה

In [ ]:
st = pd.DataFrame([{
    "method": m,
    "max_km": round(dfs[m].km.max(), 2), "mean_km": round(dfs[m].km.mean(), 2),
    "median_km": round(dfs[m].km.median(), 2), "p90_km": round(dfs[m].km.quantile(.9), 2),
    "p99_km": round(dfs[m].km.quantile(.99), 2), "max_cells": int(dfs[m].n_cells.max()),
    "mean_support": round(dfs[m].support.mean(), 1), "max_support": int(dfs[m].support.max()),
} for m in dfs])
display(st)


## 5. הסכמה בין השיטות

מודדים כמה השיטות מסכימות על קבוצת תתי-המסלולים (לפי `win_key`), עם השיטה המדויקת כייחוס:
- **Jaccard** = |חיתוך| / |איחוד|
- **Precision/Recall/F1** ביחס לשיטת הייחוס

In [ ]:
ref = set(dfs[REFERENCE].win_key)
rows = []
for m in dfs:
    if m == REFERENCE: continue
    B = set(dfs[m].win_key); inter = len(ref & B)
    P = inter/len(B) if B else 0; R = inter/len(ref) if ref else 0
    F1 = 2*P*R/(P+R) if (P+R) else 0
    rows.append({"method": m, "vs": REFERENCE, "jaccard": round(inter/len(ref|B),4),
                 "precision": round(P,4), "recall": round(R,4), "F1": round(F1,4),
                 "only_in_ref": len(ref-B), "only_in_method": len(B-ref)})
display(pd.DataFrame(rows) if rows else "only one method loaded")


## 6. Trade-off: דיוק (F1) מול משאבים (זמן/זיכרון)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar([metrics[m]["method"] for m in dfs], [metrics[m]["runtime_sec"] for m in dfs], color="steelblue")
ax[0].set_title("Runtime (sec)"); ax[0].tick_params(axis="x", rotation=20)
ax[1].bar([metrics[m]["method"] for m in dfs], [metrics[m]["struct_mem_bytes"]/1e6 for m in dfs], color="indianred")
ax[1].set_title("Aux structure memory (MB)"); ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


## מסקנות למצגת

- **נכונות**: השיטות מחזירות (כמעט) אותה קבוצת תתי-מסלולים — Jaccard גבוה מאמת את שתיהן.
- **ביצועים**: שיטת ה-Suffix Array מהירה על מכונה אחת אך אינה מבוזרת; שיטת הגרף איטית יותר אך מתרחבת לקלאסטר Spark — קריטי לדאטה המלא.
- **משאבים מקורבים**: Count-Min Sketch ו-HyperLogLog נותנים דיוק גבוה (שגיאה < ~1%) בזיכרון זעיר וקבוע.

מלאו את הטבלאות האלו עבור ההרצה על הדאטה המלא ב-DataProc — שם ההבדלים בין מבוזר ללא-מבוזר יבלטו.
